In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
ratings = pd.read_csv(
    "ratings dataset.csv",
    usecols=['userId', 'movieId', 'rating'],
    dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'}
)

movies = pd.read_csv(
    "movies dataset.csv",
    usecols=['movieId', 'title']
)

print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)


Ratings shape: (32000204, 3)
Movies shape: (87585, 2)


In [3]:
movie_counts = ratings['movieId'].value_counts()

# Keep movies with at least 200 ratings
popular_movies = movie_counts[movie_counts >= 200].index

ratings = ratings[ratings['movieId'].isin(popular_movies)]

print("After movie filtering:", ratings.shape)


After movie filtering: (30821662, 3)


In [4]:
user_counts = ratings['userId'].value_counts()

# Keep users who rated at least 100 movies
active_users = user_counts[user_counts >= 100].index

ratings = ratings[ratings['userId'].isin(active_users)]

print("After user filtering:", ratings.shape)


After user filtering: (25157166, 3)


In [5]:
data = pd.merge(ratings, movies, on='movieId')

print("Merged shape:", data.shape)
data.head()


Merged shape: (25157166, 4)


,userId,movieId,rating,title
0,1,17,4.0,Sense and Sensibility (1995)
1,1,25,1.0,Leaving Las Vegas (1995)
2,1,29,2.0,"City of Lost Children, The (Cité des enfants p..."
3,1,30,5.0,Shanghai Triad (Yao a yao yao dao waipo qiao) ...
4,1,32,5.0,Twelve Monkeys (a.k.a. 12 Monkeys) (1995)


In [6]:
movie_user_matrix = data.pivot_table(
    index='title',
    columns='userId',
    values='rating'
)

movie_user_matrix = movie_user_matrix.fillna(0)

print("Matrix shape:", movie_user_matrix.shape)


Matrix shape: (9320, 79924)


In [7]:
movie_similarity = cosine_similarity(movie_user_matrix)

movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

print("Similarity matrix created!")


Similarity matrix created!


In [11]:
def recommend_similar_movies(movie_title, num_recommendations=5):
    
    if movie_title not in movie_similarity_df.columns:
        return "Movie not found in dataset."
    
    similar_scores = movie_similarity_df[movie_title].sort_values(ascending=False)
    
    similar_scores = similar_scores.iloc[1:]
    
    return similar_scores.head(num_recommendations)


In [12]:
recommend_similar_movies("Jurassic Park (1993)")


title
Terminator 2: Judgment Day (1991)                    0.738002
Independence Day (a.k.a. ID4) (1996)                 0.724618
Forrest Gump (1994)                                  0.722271
Star Wars: Episode IV - A New Hope (1977)            0.707771
Star Wars: Episode VI - Return of the Jedi (1983)    0.696327
Name: Jurassic Park (1993), dtype: float32